## Import Libraries

In [1]:
# Import tools we need
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries ready!")

Libraries ready!


## Load Data 


In [2]:
# Load the dataset
df = pd.read_csv('../data/telco_churn.csv')

print("Dataset loaded!")
print("Rows and Columns:", df.shape)
df.head()

Dataset loaded!
Rows and Columns: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Check Data Quality

In [3]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicates
print("\nDuplicate Rows:", df.duplicated().sum())

# Check data types
print("\nData Types:")
print(df.dtypes)

Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Duplicate Rows: 0

Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
P

##  Fix TotalCharges Column

In [4]:
# TotalCharges has spaces instead of numbers
# We need to fix this

# Replace empty spaces with 0
df['TotalCharges'] = df['TotalCharges'].replace(' ', '0')

# Convert to number
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

print("TotalCharges fixed!")
print("Missing Values now:", df['TotalCharges'].isnull().sum())

TotalCharges fixed!
Missing Values now: 0


## Clean Data

In [5]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Drop customerID column — not needed for analysis
df.drop(columns=['customerID'], inplace=True)

# Convert SeniorCitizen from 0/1 to Yes/No
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print("Cleaning done!")
print("New Shape:", df.shape)

Cleaning done!
New Shape: (7043, 20)


## Add New Business Columns

In [6]:
# Create tenure bands — groups customers by how long they stayed
df['Tenure Band'] = pd.cut(df['tenure'],
                           bins=[0, 12, 24, 48, 72],
                           labels=['0-12 months',
                                   '13-24 months',
                                   '25-48 months',
                                   '49-72 months'])

# Create monthly charge bands
df['Charge Band'] = pd.cut(df['MonthlyCharges'],
                           bins=[0, 35, 65, 95, 120],
                           labels=['Low (0-35)',
                                   'Medium (35-65)',
                                   'High (65-95)',
                                   'Very High (95+)'])

# Revenue at risk — monthly charges of churned customers
df['Revenue at Risk'] = df.apply(
    lambda x: x['MonthlyCharges'] if x['Churn'] == 'Yes' else 0, axis=1)

print("New columns added!")
df.head()

New columns added!


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Tenure Band,Charge Band,Revenue at Risk
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-12 months,Low (0-35),0.00
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,...,No,One year,No,Mailed check,56.95,1889.50,No,25-48 months,Medium (35-65),0.00
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-12 months,Medium (35-65),53.85
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48 months,Medium (35-65),0.00
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-12 months,High (65-95),70.70


## Business Summary

In [7]:
total_customers  = len(df)
churned          = df[df['Churn'] == 'Yes'].shape[0]
churn_rate       = round(churned / total_customers * 100, 2)
revenue_at_risk  = round(df['Revenue at Risk'].sum(), 2)
avg_tenure       = round(df['tenure'].mean(), 1)
avg_charges      = round(df['MonthlyCharges'].mean(), 2)

print("=" * 45)
print("BUSINESS SUMMARY")
print("=" * 45)
print(f"Total Customers   : {total_customers}")
print(f"Churned Customers : {churned}")
print(f"Churn Rate        : {churn_rate}%")
print(f"Revenue at Risk   : ${revenue_at_risk}")
print(f"Avg Tenure        : {avg_tenure} months")
print(f"Avg Monthly Charge: ${avg_charges}")

BUSINESS SUMMARY
Total Customers   : 7043
Churned Customers : 1869
Churn Rate        : 26.54%
Revenue at Risk   : $139130.85
Avg Tenure        : 32.4 months
Avg Monthly Charge: $64.76


## Save Cleaned Data

In [10]:
# Save cleaned file
df.to_csv('../data/churn_cleaned.csv', index=False)
print("Cleaned file saved!")
print("Final Shape:", df.shape)

Cleaned file saved!
Final Shape: (7043, 23)
